Testando BGE-M3

In [57]:
from FlagEmbedding import BGEM3FlagModel
from FlagEmbedding import FlagReranker
from qdrant_client.models import Prefetch, FusionQuery, Fusion, models
from qdrant_client import QdrantClient, models
from qdrant_client.models import Distance, VectorParams, SparseVectorParams
from qdrant_client.models import PointStruct, SparseVector
import uuid
import pprint
from bson import ObjectId
import json
import io
import os
import boto3
import pandas as pd

Conectar banco MinIO

In [2]:
def get_parquet_file(bucket, file_name):
    endpoint = os.getenv("MINIO_ENDPOINT", "http://localhost:9000")

    s3 = boto3.client(
        "s3",
        endpoint_url=endpoint,
        aws_access_key_id=os.getenv("MINIO_ACCESS_KEY"),
        aws_secret_access_key=os.getenv("MINIO_SECRET_KEY"),
    )

    response = s3.get_object(
        Bucket=bucket,
        Key=file_name
    )

    return pd.read_parquet(
        io.BytesIO(response["Body"].read())
    )

Obter dados do env

In [ ]:
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
professores_dados = get_parquet_file("gold", "professores_unificados.parquet")

In [5]:
disciplinas_dados = get_parquet_file("silver", "componentes_curriculares.parquet")

In [6]:
ics_dados = get_parquet_file("silver", "trabalhos_ic_periodicos.parquet")

In [7]:
print(professores_dados.columns)

Index(['nome', 'idLattes', 'siape', 'resumo', 'formacaoAcademicaProfissional',
       'areasInteresse', 'competencias', 'producoes', 'producaoDocenteSigaa',
       'projetos', 'orientacoes', 'disciplinasSigaa',
       'trabalhosIniciacaoCientifica'],
      dtype='object')


In [8]:
print(disciplinas_dados.columns)

Index(['idSigaa', 'codigo', 'nome', 'cargaHoraria', 'ementa'], dtype='object')


In [9]:
print(ics_dados.columns)

Index(['id', 'titulo', 'autores', 'resumo', 'palavrasChaves', 'ano'], dtype='object')


In [10]:
ics_dados = ics_dados.to_dict("records")

In [11]:
professores_dados = professores_dados.to_dict("records")

In [181]:
disciplinas_dados = disciplinas_dados.to_dict("records")

Criação do perfil do professor

In [21]:
def montar_texto_producao(producao):
    partes = []

    if producao.get("titulo"):
        partes.append(f"Título: {producao['titulo']}")

    if producao.get("resumo"):
        partes.append(f"Resumo: {producao['resumo']}")

    if producao.get("palavrasChaves").any():
        partes.append(
            f"Palavras-chave: {', '.join(producao['palavrasChaves'])}"
        )

    return "\n".join(partes)

In [13]:
def montar_texto_competencias(competencias):
    partes = []

    areas = competencias.get("areas")

    if areas is not None and len(areas) > 0:
        textos_areas = []

        for area in areas:
            campos = []

            if area.get("grande_area"):
                campos.append(f"Grande área: {area['grande_area']}")

            if area.get("area"):
                campos.append(f"Área: {area['area']}")

            if area.get("subarea"):
                campos.append(f"Subárea: {area['subarea']}")

            if area.get("especialidade"):
                campos.append(f"Especialidade: {area['especialidade']}")

            if campos:
                textos_areas.append(". ".join(campos))

        if textos_areas:
            partes.append(
                "Áreas de conhecimento:\n" +
                "\n".join(textos_areas)
            )

    linhas = competencias.get("linhas_pesquisa")

    if linhas is not None and len(linhas) > 0:
        partes.append(
            "Linhas de pesquisa: " + ", ".join(linhas)
        )

    palavras_chave = competencias.get("palavras_chave")

    if palavras_chave is not None and len(palavras_chave) > 0:
        partes.append(
            "Palavras-chave: " + ", ".join(palavras_chave)
        )

    return "\n\n".join(partes)

In [14]:
def montar_texto_perfil_professor(perfil):

    partes = []
    
    if perfil.get("resumo"):
        partes.append(f"Resumo: {perfil['resumo']}")
    
    if perfil.get("formacaoAcademicaProfissional"):
        partes.append(
            f"Formação Profissional Acadêmica: "
            f"{perfil['formacaoAcademicaProfissional']}"
        )

    if perfil.get("areasInteresse").any():
        partes.append(
            f"Áreas de Interesse: "
            f"{', '.join(perfil['areasInteresse'])}"
        )    
    if perfil.get("competencias"):
        competencias_texto = montar_texto_competencias(
            perfil["competencias"]
        )
        if competencias_texto:
            partes.append(competencias_texto)
    
    return "\n".join(partes)

In [193]:
def montar_texto_disciplina(disciplina):
    partes = []

    if disciplina.get("nome"):
        partes.append(f"Nome: {disciplina['nome']}")

    if disciplina.get("ementa"):
        partes.append(f"Ementa: {disciplina['ementa']}")

    return "\n".join(partes)

BGE-M3

In [15]:
# bge-m3 vai ficar na mémoria da nossa aplicação
def carregar_bgeM3():
    return BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)

def rodar_bgeM3(modelo, texto):
    embedding = {}
    resultado = modelo.encode(
        texto,
        return_dense=True, 
        return_sparse=True
    )

    embedding = {
        "dense": resultado["dense_vecs"],
        "sparse": resultado["lexical_weights"]
    }

    return embedding

def obter_denso(perfis_transformados):
    return perfis_transformados['dense_vecs']
    

In [16]:
modelo = carregar_bgeM3()

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Criação dos Hash dos textos de Embedding

In [18]:
def gerar_hash(componentes):   
    json_normalizado = json.dumps(
        componentes,
        sort_keys=True,
        ensure_ascii=False,
        separators=(",", ":")
    )
    return hashlib.sha256(transformar_bytes(json_normalizado)).hexdigest()

In [17]:
import hashlib

def transformar_bytes(texto : str) -> bytes:
    t = texto
    return t.encode("utf-8")

Componentes (por enquanto) considerados para a busca :
- Perfil professor (Resumo, Formação Acadêmica Profissional, Áreas de interesse, Competências)
- Iniciação Científica
- Disciplinas

Criação do perfil dos conteúdos dos componentes

In [ ]:
def criar_perfils_trabalhos(trabalhos) -> list[dict]:
    trabalhos_texto = []
    for trabalho in trabalhos:
        texto_embedding = montar_texto_producao(trabalho)
        trabalho_transformado = {
            "id" : trabalho['id'], 
            "hash" : gerar_hash(texto_embedding),
            "texto_embedding": texto_embedding,
            "metadata" : {
               "titulo": trabalho['titulo'],
               "ano" : int(trabalho['ano']),
               "tipo": trabalho.get("tipo")
            }
        }
        trabalhos_texto.append(trabalho_transformado)
        
    return trabalhos_texto


In [22]:
ics_perfis_busca = criar_perfils_trabalhos(ics_dados)

In [24]:
ics_valores_denso_lexical= {}  
for ic in ics_perfis_busca:
        
    texto_transformado = rodar_bgeM3(modelo,ic.get("texto_embedding"))

    ics_valores_denso_lexical[ic["id"]] = texto_transformado

In [26]:
ics_valores_denso_lexical['9b3189cbc1da']

{'dense': array([-0.04400245,  0.02993284, -0.00196428, ..., -0.02217614,
        -0.02278105, -0.0018368 ], shape=(1024,), dtype=float32),
 'sparse': defaultdict(int,
             {'211025': np.float32(0.10696598),
              '122461': np.float32(0.10371503),
              '866': np.float32(0.13957539),
              '124661': np.float32(0.14100933),
              '241': np.float32(0.06076681),
              '55991': np.float32(0.13232352),
              '7146': np.float32(0.10805798),
              '276': np.float32(0.08453494),
              '43351': np.float32(0.12482357),
              '52069': np.float32(0.043306984),
              '117772': np.float32(0.07535648),
              '62958': np.float32(0.122521356),
              '8575': np.float32(0.12503597),
              '122359': np.float32(0.16912784),
              '36321': np.float32(0.09868437),
              '89944': np.float32(0.08621602),
              '6': np.float32(0.011776261),
              '49877': np.float32(0.1

In [30]:
def criar_perfils_professor(professores) -> list[dict]:
    perfis_texto = []
    for professor in professores:
        texto_embedding = montar_texto_perfil_professor(professor)
        professor_transformado = {
            "id" : professor['siape'], 
            "hash" : gerar_hash(texto_embedding),
            "texto_embedding": texto_embedding,
            "metadata" : {
               "nome": professor['nome'],
            }
        }
        perfis_texto.append(professor_transformado)

    return perfis_texto


In [31]:
professores_busca_perfis = criar_perfils_professor(professores_dados)

In [32]:
professores_busca_perfis

[{'id': '2234207',
  'hash': 'b5e1a2c4f3aba88a43fdeea157761b94dc1ae1c190371aee708d9f87d8ce4f2a',
  'texto_embedding': 'Resumo: Possui graduação em Engenharia Elétrica pela Universidade Federal de Itajubá (2010), mestrado e doutorado em Engenharia de Produção pela Universidade Federal de Itajubá (2015/2020). É especialista em Engenharia Clínica, trabalhou na área por 4 anos, com nos seguintes temas: inventário, seis sigma, interferências eletromagnéticas, equipamentos médicos, usabilidade e engenharia de fatores humanos. Também possui experiência na área de Engenharia de Produção em indústria de manufatura intensiva, na área de qualidade melhorias, com aplicação de Kaizens e projetos Seis Sigma. É professora da UNIFEI nas áreas de Eletrônica Analógica, Instrumentação Biomédica, Metodologia Científica e Usabilidade. É pesquisadora no programa Desenvolvimento Sociedade e Tecnologias e no Laboratório de Usabilidade e Fatores Humanos da UNIFEI\nFormação Profissional Acadêmica: Doutorado\nÁr

In [33]:
professores_valores_denso_lexical = {}  
for professor in professores_busca_perfis:
        
    texto_transformado = rodar_bgeM3(modelo,professor.get("texto_embedding"))

    professores_valores_denso_lexical[professor["id"]] = texto_transformado

In [194]:
def criar_perfils_disciplinas(disciplinas) -> list[dict]:
    disciplina_texto = []
    for disciplina in disciplinas:
        texto_embedding = montar_texto_disciplina(disciplina)
        trabalho_transformado = {
            "id" : disciplina['idSigaa'], 
            "hash" : gerar_hash(texto_embedding),
            "texto_embedding": texto_embedding,
            "metadata" : {
               "nome": disciplina['nome'],
               "codigo": disciplina["codigo"]

            }
        }
        disciplina_texto.append(trabalho_transformado)
        
    return disciplina_texto

In [195]:
disciplinas_perfil_busca = criar_perfils_disciplinas(disciplinas_dados)

In [197]:
disciplinas_valores_denso_lexical = {}  
for disciplina in disciplinas_perfil_busca:

    texto_transformado = rodar_bgeM3(modelo,disciplina.get("texto_embedding"))

    disciplinas_valores_denso_lexical[disciplina["id"]] = texto_transformado

Qdrant

In [ ]:
import uuid

def qdrant_id(id):
    return str(uuid.uuid5(uuid.NAMESPACE_URL, f"{id}"))

In [ ]:
def conectar_qdrant(url):
    return QdrantClient(url=url)

def criar_banco_qdrant(client_qdrant, nome):
    client_qdrant.create_collection(
        collection_name= nome,
        vectors_config={
            "dense": VectorParams(size=1024, distance=Distance.COSINE)},
        sparse_vectors_config={
            "sparse": SparseVectorParams(),
        },
    )

def atualizar_perfis(perfis_transformados, client_qdrant):

    professores_para_atualizar = []

    for perfil in perfis_transformados:

        professor_id = qdrant_id(perfil.get("id"))
        hash_atual = perfil.get("hash")


        busca = client_qdrant.retrieve(
            collection_name="professores",
            ids=[professor_id],
            with_payload=True
        )
        if not busca:
            professores_para_atualizar.append({
                "perfil": perfil,
            })
            continue
        
        payload_qdrant = busca[0].payload

        hash_salvo = payload_qdrant.get("hash")

        if hash_salvo != hash_atual:
                professores_para_atualizar.append({"perfil": perfil})

    return professores_para_atualizar

In [ ]:
def adicionar_trabalhos_qdrant(client_qdrant, conteudo, resultados, collection = ""):
    points = []
    for objeto in conteudo:
        
        objeto_id  = objeto.get('id')
        
        resultado = resultados[objeto_id]

        sparse = resultado["sparse"]

        sparse_indices = [int(k) for k in sparse.keys()]
        sparse_values = [float(v) for v in sparse.values()]
            
        dense = resultado["dense"]
            
        points.append(
            PointStruct(
                id=qdrant_id(objeto.get("id")),
                vector={
                    "dense": dense.tolist(),
                    "sparse": SparseVector(indices=sparse_indices, values=sparse_values),
                },
                # metadados
                payload={
                    "id": objeto.get("id"),
                    "titulo":  objeto.get("metadata", {}).get("titulo"),
                    "ano": objeto.get("metadata", {}).get("ano"),
                    "hash":objeto.get("hash")
                }
            )
        )
            
    client_qdrant.upsert(collection_name=collection, points=points)


In [235]:
def adicionar_professores_qdrant(client_qdrant, professores, resultados, collection = ""):
    points = []
    for objeto in professores:
        
        objeto_id  = objeto.get('id')
        
        resultado = resultados[objeto_id]

        sparse = resultado["sparse"]

        sparse_indices = [int(k) for k in sparse.keys()]
        sparse_values = [float(v) for v in sparse.values()]
            
        dense = resultado["dense"]
            
        points.append(
            PointStruct(
                id=qdrant_id(objeto.get("id")),
                vector={
                    "dense": dense.tolist(),
                    "sparse": SparseVector(indices=sparse_indices, values=sparse_values),
                },
                # metadados
                payload={
                    "id": objeto.get("id"),
                    "nome":  objeto.get("metadata", {}).get("nome"),
                    "hash" :  objeto.get("hash")
                }
            )
        )
            
    client_qdrant.upsert(collection_name=collection, points=points)


In [ ]:
def adicionar_disciplinas_qdrant(
    client_qdrant,
    conteudo,
    resultados,
    collection="",
    batch_size=50
):
    points = []

    for objeto in conteudo:

        objeto_id = objeto.get("id")
        resultado = resultados[objeto_id]

        sparse = resultado["sparse"]

        sparse_indices = [int(k) for k in sparse.keys()]
        sparse_values = [float(v) for v in sparse.values()]

        dense = resultado["dense"]

        points.append(
            PointStruct(
                id=qdrant_id(objeto_id),
                vector={
                    "dense": dense.tolist(),
                    "sparse": SparseVector(
                        indices=sparse_indices,
                        values=sparse_values
                    ),
                },
                payload={
                    "id": objeto_id,
                    "nome": objeto.get("metadata", {}).get("nome"),
                    "hash": objeto.get("hash")
                }
            )
        )
    #Enviando aos poucos já que tem muitas disciplinas
    for i in range(0, len(points), batch_size):
        batch = points[i:i + batch_size]

        client_qdrant.upsert(
            collection_name=collection,
            points=batch,
            wait=True
        )

In [39]:
# aplica filtro do payload antes da busca (colocar antes de adicionar dados)
def criacao_payload_filter(client, collection, field, tipo = models.PayloadSchemaType.INTEGER):
    client.create_payload_index(
        collection_name=collection,
        field_name=field,
        field_schema=tipo,
    )

Conectando ao Qdrant no Docket

In [31]:
client_qdrant = conectar_qdrant("http://localhost:6333")

In [198]:
criar_banco_qdrant(client_qdrant, "ics")


KeyboardInterrupt



In [ ]:
adicionar_trabalhos_qdrant(client_qdrant, ics_transformadas, ics_perfis_busca, collection = "ics")

In [171]:
criar_banco_qdrant(client_qdrant, "professores")

In [ ]:
adicionar_professores_qdrant(client_qdrant, professores_busca_perfis, professores_valores_denso_lexical, collection = "professores")

Qdrant cloud

In [203]:
from qdrant_client import QdrantClient, models
import os

client = QdrantClient(url=os.getenv("QDRANT_URL"), api_key=os.getenv("QDRANT_API_KEY"))

In [204]:
# Quick health check
collections = client.get_collections()
print(f"Connected to Qdrant Cloud: {len(collections.collections)} collections")

Connected to Qdrant Cloud: 3 collections


In [65]:
criar_banco_qdrant(client,"professores")

In [66]:
adicionar_professores_qdrant(client,professores_busca_perfis,professores_valores_denso_lexical, collection="professores")

In [238]:
criar_banco_qdrant(client,"iniciacao_cientifica")

In [237]:
client.delete_collection("iniciacao_cientifica")

True

In [239]:
criacao_payload_filter(client, "iniciacao_cientifica", "ano")

In [240]:
adicionar_trabalhos_qdrant(client,ics_perfis_busca , ics_valores_denso_lexical, collection = "iniciacao_cientifica")

In [270]:
client.delete_collection("disciplinas")

True

In [271]:
criar_banco_qdrant(client,"disciplinas")

In [272]:
adicionar_disciplinas_qdrant(client,disciplinas_perfil_busca , disciplinas_valores_denso_lexical, collection = "disciplinas")

Enviando 0 - 50 de 1484
Enviando 50 - 100 de 1484
Enviando 100 - 150 de 1484
Enviando 150 - 200 de 1484
Enviando 200 - 250 de 1484
Enviando 250 - 300 de 1484
Enviando 300 - 350 de 1484
Enviando 350 - 400 de 1484
Enviando 400 - 450 de 1484
Enviando 450 - 500 de 1484
Enviando 500 - 550 de 1484
Enviando 550 - 600 de 1484
Enviando 600 - 650 de 1484
Enviando 650 - 700 de 1484
Enviando 700 - 750 de 1484
Enviando 750 - 800 de 1484
Enviando 800 - 850 de 1484
Enviando 850 - 900 de 1484
Enviando 900 - 950 de 1484
Enviando 950 - 1000 de 1484
Enviando 1000 - 1050 de 1484
Enviando 1050 - 1100 de 1484
Enviando 1100 - 1150 de 1484
Enviando 1150 - 1200 de 1484
Enviando 1200 - 1250 de 1484
Enviando 1250 - 1300 de 1484
Enviando 1300 - 1350 de 1484
Enviando 1350 - 1400 de 1484
Enviando 1400 - 1450 de 1484
Enviando 1450 - 1484 de 1484


Aluno fez preencheu o formulário

In [273]:
aluno_texto = "Gostaria de desenvolver um projeto com inteligência artificial e programação embarcada"
aluno = modelo.encode(aluno_texto, return_dense=True, return_sparse=True)
embedding_aluno = aluno['dense_vecs']
lexical_aluno = aluno['lexical_weights']

In [241]:
def busca_hibrida_qdrant(client_qdrant,collection,dense,sparse, limit=50):
    sparse_query = SparseVector(
        indices=[int(k) for k in sparse.keys()],
        values=[float(v) for v in sparse.values()],
    )

    prefetch = [
        Prefetch(
            query=dense.tolist(),
            using="dense",
            limit=limit
        ),
        Prefetch(
            query=sparse_query,
            using="sparse",
            limit=limit
        )
    ]

    results = client_qdrant.query_points(
        collection_name=collection,
        prefetch=prefetch,
        query=FusionQuery(
            fusion=Fusion.RRF
        ),
        limit=limit,
        with_payload=True
    )
    return results.points

In [242]:
busca_realizada_ics = busca_hibrida_qdrant(client,"iniciacao_cientifica", embedding_aluno, lexical_aluno, limit=50)

In [243]:
for point in busca_realizada_ics:
    print(
        point.payload["id"],
        point.payload["titulo"],
        point.score
    )

f92d111e5241 Estruturação de hardware e software para desenvolvimento de um agv 0.6111111
330dbb5265cd DETECÇÃO DE PRAGAS EM LAVOURAS COM REDES NEURAIS CONVOLUCIONAIS EM SISTEMAS EMBARCADOS 0.5416667
6d9b570e8513 ESTUDO DE SISTEMAS EMBARCADOS NA AGRICULTURA DIGITAL 0.4166667
135b69c36144 DESENVOLVIMENTO DE RECURSOS EDUCACIONAIS ABERTOS SOBRE MICRORREDES 0.41025642
8f39dd8786ae Desenvolvimento do firmware para acionamento e aquisição de dados para veículos autônomos 0.39285713
9617879d2744 Estudo de algoritmo de controle de trajetórias de uma Aeronave Remotamente Pilotada 0.36904764
f0890bd4eb6a Estudo e desenvolvimento de uma microarquitetura RISC-V baseado no ISA RV32I. 0.325
d654c8a463bc Segmentação Semântica Integrada ao ROS2 para Aplicações em Veículos Autônomos 0.23376624
a23a21b54ceb Sistema de armazenamento de energia combinado com geração fotovoltaica operando como formador de rede ou supridor de rede 0.22222222
92d0c1e746e0 APLICAÇÃO DE TÉCNICAS DE INTELIGÊNCIA ARTIFICIAL PARA

In [220]:
busca_realizada_perfis_professores = busca_hibrida_qdrant(client,"professores",embedding_aluno,lexical_aluno,limit = 50)

In [221]:
for point in busca_realizada_perfis_professores:
    print(
        point.payload["id"],
        point.payload["nome"],
        point.score
    )

1545021 CARLOS HENRIQUE VALERIO DE MORAES 0.625
1682528 THATYANA DE FARIA PIOLA SERAPHIM 0.54761904
1967206 LUCIANO BERTINI 0.5
1732597 RODRIGO MAXIMIANO ANTUNES DE ALMEIDA 0.4761905
1374755 DECIO RENNO DE MENDONCA FARIA 0.35
3529390 BRENO DE OLIVEIRA RENO 0.3125
394747 LUIZ EDIVAL DE SOUZA 0.25555557
394714 JOSE ALBERTO FERREIRA FILHO 0.23809525
3443612 BARBARA PIMENTA CAETANO 0.22619048
1097837 GABRIEL CIRAC MENDES DE SOUZA 0.2
1543328 ENZO SERAPHIM 0.1625
395119 TALES CLEBER PIMENTA 0.15725806
1457656 EDMILSON MARMO MOREIRA 0.15277778
2195171 ROBSON LUIZ MORENO 0.15
2642873 KLEBER ROBERTO DA SILVA SANTOS 0.13968255
2019312 ODILON DE OLIVEIRA DUTRA 0.13247864
3150657 GABRIEL ANTONIO FANELLI DE SOUZA 0.12692308
1999579 JOAO PAULO REUS RODRIGUES LEITE 0.124242425
1862488 JEREMIAS BARBOSA MACHADO 0.124060154
6394694 EGON LUIZ MULLER JUNIOR 0.11948052
1310498 ANDRE BERNARDI 0.116666675
1560417 LUIS HENRIQUE DE CARVALHO FERREIRA 0.09882353
1666035 LEONARDO BRESEGHELLO ZOCCAL 0.09453782
18

In [341]:
busca_realizada_disciplinas = busca_hibrida_qdrant(client,"disciplinas",embedding_aluno,lexical_aluno,limit = 100)

In [342]:
for point in busca_realizada_disciplinas:
    print(
        point.payload["id"],
        point.payload["nome"],
        point.score
    )

5651 PROGRAMAÇÃO PARA SISTEMAS EMBARCADOS 0.75
5300 LABORATÓRIO DE PROGRAMAÇÃO EMBARCADA 0.625
5295 PROGRAMAÇÃO EMBARCADA 0.4761905
5310 INTELIGÊNCIA ARTIFICIAL 0.35833335
5314 LABORATÓRIO DE INTELIGÊNCIA ARTIFICIAL 0.28125
5673 APRENDIZADO BASEADO EM PROJETO III 0.2361111
29724 Inteligência Artificial 0.2212766
7677 Programação Avançada 0.2
29725 Inteligência Artificial (Prática) 0.1969697
30006 Programação Avançada 0.16666667
7265 MANUFATURA INTEGRADA POR COMPUTADOR 0.14285715
5565 Sistemas Embarcados com ARM 7 0.124390244
47044 INTELIGÊNCIA ARTIFICIAL APLICADA 0.11111111
46958 PROJETO DE SOFTWARE 0.10240334
29452 Programação Aplicada 0.10220126
46945 PROJETO INTEGRADOR DE AUTOMAÇÃO 0.1
7667 PROGRAMAÇÃO ORIENTADA A OBJETOS 0.09090909
5676 LABORATÓRIO DE PROGRAMAÇÃO 0.08703704
49442 INTELIGÊNCIA ARTIFICIAL II 0.083333336
5243 PROGRAMAÇÃO APLICADA 0.082633056
5245 LABORATÓRIO DE PROGRAMAÇÃO APLICADA 0.077030815
5240 PROGRAMAÇÃO ORIENTADA A OBJETOS 0.07692308
581 PROJETO DE CIRCUITOS IN

Relacionando as iniciações científicas com seus professores

Faz em Dataframe pela possibilidade de realizar merge

In [294]:
def relacionar_dados(professores, relacionado = ""):
    relacoes = []

    for professor in professores:
        siape = professor.get("siape")
        nome = professor.get("nome")

        trabalhos = professor.get(relacionado, [])

        for trabalho in trabalhos:
            relacoes.append({
                "professor_id": siape,
                "nome": nome,
                "objeto_id": trabalho
            })
            
    return pd.DataFrame(relacoes)

In [295]:
relacao_professor_ic = relacionar_dados(professores_dados, "trabalhosIniciacaoCientifica")

In [ ]:
relacao_professor_ic

,professor_id,nome,objeto_id
0,2127183,BRUNO TARDIOLE KUEHNE,f001be6c8d33
1,2127183,BRUNO TARDIOLE KUEHNE,a0d1dae32993
2,2127183,BRUNO TARDIOLE KUEHNE,1fe0a29a59a3
3,1810770,DANILO HENRIQUE SPADOTI,6eb11d095a65
4,1810770,DANILO HENRIQUE SPADOTI,0af079870b1f
5,1810770,DANILO HENRIQUE SPADOTI,843757a0e743
6,1810770,DANILO HENRIQUE SPADOTI,f55732890b00
7,1810770,DANILO HENRIQUE SPADOTI,1904316495e5
8,1634986,EDVARD MARTINS DE OLIVEIRA,6d9b570e8513
9,1634986,EDVARD MARTINS DE OLIVEIRA,276846a8b564


In [297]:
def criar_dataframe_resultados_trabalhos(resultados_trabalhos):
    resultados = []

    for point in resultados_trabalhos:
        resultados.append({
            "objeto_id": str(point.payload["id"]),
            "score": point.score
        })

    return pd.DataFrame(resultados)

In [298]:
def associar_scores_a_professores(resultados_trabalhos,relacao):
    trabalhos_df = criar_dataframe_resultados_trabalhos(
        resultados_trabalhos
    )

    resultado = relacao.merge(
        trabalhos_df,
        on="objeto_id",
        how="inner"
    )

    return resultado.sort_values(
        by="score",
        ascending=False
    ).reset_index(drop=True)

Associar as iniciações científicas buscadas com os professores correspondentes

In [299]:
resultado_professor_trabalho_unico = associar_scores_a_professores(busca_realizada_ics, relacao_professor_ic)

In [300]:
associar_scores_a_professores(busca_realizada_ics, relacao_professor_ic)

,professor_id,nome,objeto_id,score
0,1634986,EDVARD MARTINS DE OLIVEIRA,6d9b570e8513,0.416667
1,2195502,GUSTAVO DELLA COLLETTA,f0890bd4eb6a,0.325000
2,2256795,ROBSON BAUWELZ GONZATTI,92d0c1e746e0,0.210145
3,2127183,BRUNO TARDIOLE KUEHNE,f001be6c8d33,0.155556
4,3297076,WILSON CESAR SANT'ANA,198d41e77b76,0.133838
5,2642873,KLEBER ROBERTO DA SILVA SANTOS,ad87b6f82193,0.121795
6,2256795,ROBSON BAUWELZ GONZATTI,d20805e76bba,0.076316
7,3297076,WILSON CESAR SANT'ANA,e5bd86f84b37,0.071429
8,6394694,EGON LUIZ MULLER JUNIOR,efc40bb6e0cf,0.063508
9,2195502,GUSTAVO DELLA COLLETTA,618c08cdde4c,0.058824


Unir diferentes trabalhos de um mesmo professor 
- Pega os 3 com maiores scores e faz a média

In [301]:
def calcular_score_trabalhos(resultado):

    scores = (
        resultado
        .sort_values(
            by=["professor_id", "score"],
            ascending=[True, False]
        )
        .groupby("professor_id")
        .head(3)
        .groupby("professor_id")["score"]
        .mean()
        .reset_index()
        .rename(columns={
            "score": "work_score"
        })
    )

    return scores

In [ ]:
score_trabalhos_por_professor = calcular_score_trabalhos(resultado_professor_trabalho_unico)

In [303]:
calcular_score_trabalhos(resultado_professor_trabalho_unico)

,professor_id,work_score
0,1634986,0.416667
1,1810770,0.037879
2,2127183,0.155556
3,2195502,0.191912
4,2256795,0.143230
5,2642873,0.121795
6,3150657,0.038462
7,3297076,0.102633
8,6394694,0.056754


Score do perfil dos professores

In [111]:
def criar_dataframe_resultados_professores(resultados):
    dados = []

    for ponto in resultados:
        if not ponto.payload:
            continue

        dados.append({
            "professor_id": ponto.payload["id"],
            "professor_nome": ponto.payload["nome"],
            "professor_score": ponto.score
        })

    return pd.DataFrame(dados)

In [ ]:
scores_perfis_professores = (criar_dataframe_resultados_professores(busca_realizada_perfis_professores))

In [ ]:
criar_dataframe_resultados_professores(busca_realizada_perfis_professores)

,professor_id,professor_nome,professor_score
0,1545021,CARLOS HENRIQUE VALERIO DE MORAES,0.625000
1,1682528,THATYANA DE FARIA PIOLA SERAPHIM,0.547619
2,1967206,LUCIANO BERTINI,0.500000
3,1732597,RODRIGO MAXIMIANO ANTUNES DE ALMEIDA,0.476191
4,1374755,DECIO RENNO DE MENDONCA FARIA,0.350000
5,3529390,BRENO DE OLIVEIRA RENO,0.312500
6,394747,LUIZ EDIVAL DE SOUZA,0.255556
7,394714,JOSE ALBERTO FERREIRA FILHO,0.238095
8,3443612,BARBARA PIMENTA CAETANO,0.226190
9,1097837,GABRIEL CIRAC MENDES DE SOUZA,0.200000


Unir disciplinas e professores

In [343]:
relacao_professor_disciplina = relacionar_dados(professores_dados, "disciplinasSigaa")

In [344]:
relacao_professor_disciplina

,professor_id,nome,objeto_id
0,2234207,ANA PAULA SIQUEIRA SILVA DE ALMEIDA,175
1,2234207,ANA PAULA SIQUEIRA SILVA DE ALMEIDA,29953
2,2234207,ANA PAULA SIQUEIRA SILVA DE ALMEIDA,45429
3,2234207,ANA PAULA SIQUEIRA SILVA DE ALMEIDA,46570
4,2234207,ANA PAULA SIQUEIRA SILVA DE ALMEIDA,46576
...,...,...,...
1431,3297076,WILSON CESAR SANT'ANA,46928
1432,3297076,WILSON CESAR SANT'ANA,46960
1433,3297076,WILSON CESAR SANT'ANA,5671
1434,3297076,WILSON CESAR SANT'ANA,5693


In [345]:
resultado_professor_disciplina = associar_scores_a_professores(busca_realizada_disciplinas, relacao_professor_disciplina)

In [346]:
resultado_professor_disciplina

,professor_id,nome,objeto_id,score
0,1732597,RODRIGO MAXIMIANO ANTUNES DE ALMEIDA,5651,0.750000
1,1374755,DECIO RENNO DE MENDONCA FARIA,5300,0.625000
2,1545021,CARLOS HENRIQUE VALERIO DE MORAES,5300,0.625000
3,1967206,LUCIANO BERTINI,5300,0.625000
4,1732597,RODRIGO MAXIMIANO ANTUNES DE ALMEIDA,5300,0.625000
...,...,...,...,...
110,1457656,EDMILSON MARMO MOREIRA,29660,0.019231
111,1310498,ANDRE BERNARDI,5296,0.019231
112,1457656,EDMILSON MARMO MOREIRA,29510,0.018868
113,1999579,JOAO PAULO REUS RODRIGUES LEITE,29510,0.018868


In [305]:
def calcular_score_disciplinas(resultado):

    scores = (
        resultado
        .sort_values(
            by=["professor_id", "score"],
            ascending=[True, False]
        )
        .groupby("professor_id")
        .head(3)
        .groupby("professor_id")["score"]
        .mean()
        .reset_index()
        .rename(columns={
            "score": "disciplina_score"
        })
    )

    return scores

In [347]:
score_disciplinas = calcular_score_disciplinas(resultado_professor_disciplina)

In [348]:
score_disciplinas

,professor_id,disciplina_score
0,1062611,0.037319
1,1097837,0.324128
2,1310498,0.087288
3,1374755,0.550595
4,1457656,0.041806
5,1543328,0.072437
6,1545021,0.404101
7,1560417,0.236111
8,1634986,0.023256
9,1666035,0.025641


Junção dos scores obtidos

In [ ]:
def combinar_scores(
    scores_trabalhos,
    scores_professores,
    scores_disciplinas,
    peso_trabalhos=0.5,
    peso_professor=0.2,
    peso_disciplinas=0.3
):
    resultado = (
        scores_trabalhos
        .merge(
            scores_professores,
            on="professor_id",
            how="outer"
        )
        .merge(
            scores_disciplinas,
            on="professor_id",
            how="outer"
        )
    )

    resultado["work_score"] = (
        resultado["work_score"].fillna(0)
    )

    resultado["professor_score"] = (
        resultado["professor_score"].fillna(0)
    )

    resultado["disciplina_score"] = (
        resultado["disciplina_score"].fillna(0)
    )

    resultado["score_final"] = (
        peso_trabalhos * resultado["work_score"]
        +
        peso_professor * resultado["professor_score"]
        +
        peso_disciplinas * resultado["disciplina_score"]
    )

    return resultado.sort_values(
        "score_final",
        ascending=False
    )

In [350]:
score_final_professores = combinar_scores(score_trabalhos_por_professor,scores_perfis_professores,score_disciplinas)

In [351]:
combinar_scores(score_trabalhos_por_professor,scores_perfis_professores,score_disciplinas)

,professor_id,work_score,professor_nome,professor_score,disciplina_score,score_final
11,1732597,0.000000,RODRIGO MAXIMIANO ANTUNES DE ALMEIDA,0.476191,0.617063,0.280357
17,1967206,0.000000,LUCIANO BERTINI,0.500000,0.550595,0.265179
6,1545021,0.000000,CARLOS HENRIQUE VALERIO DE MORAES,0.625000,0.404101,0.246230
3,1374755,0.000000,DECIO RENNO DE MENDONCA FARIA,0.350000,0.550595,0.235179
8,1634986,0.416667,EDVARD MARTINS DE OLIVEIRA,0.077961,0.023256,0.230902
1,1097837,0.000000,GABRIEL CIRAC MENDES DE SOUZA,0.200000,0.324128,0.137238
10,1682528,0.000000,THATYANA DE FARIA PIOLA SERAPHIM,0.547619,0.043478,0.122567
22,2195502,0.191912,GUSTAVO DELLA COLLETTA,0.049451,0.028068,0.114266
20,2127183,0.155556,BRUNO TARDIOLE KUEHNE,0.072482,0.050089,0.107301
29,2642873,0.121795,KLEBER ROBERTO DA SILVA SANTOS,0.139683,0.027983,0.097229


Busca Densa

In [ ]:
def busca_densa (embedding_aluno, collection ="", limit = 10):
    results = client_qdrant.query_points(
        collection,
        query=embedding_aluno,
        using="dense",
        limit=limit,
    )
    return results

resultado_densa_perfil = busca_densa(embedding_aluno)
resultado_densa_disciplinas = busca_densa(embedding_aluno)
resultado_densa_ics = busca_densa(embedding_aluno)

In [ ]:
ranking_densa = []
for point in resultado_densa.points:
    print(point.payload["nome"], point.score)
    
    ranking_densa.append(point.payload["nome"], point.score)

Reranking

In [352]:
FINAL_PROFESSOR_K = 15
candidatos_reranking = score_final_professores.head(FINAL_PROFESSOR_K)

In [488]:
candidatos_reranking

,professor_id,work_score,professor_nome,professor_score,disciplina_score,score_final
11,1732597,0.000000,RODRIGO MAXIMIANO ANTUNES DE ALMEIDA,0.476191,0.617063,0.280357
17,1967206,0.000000,LUCIANO BERTINI,0.500000,0.550595,0.265179
6,1545021,0.000000,CARLOS HENRIQUE VALERIO DE MORAES,0.625000,0.404101,0.246230
3,1374755,0.000000,DECIO RENNO DE MENDONCA FARIA,0.350000,0.550595,0.235179
8,1634986,0.416667,EDVARD MARTINS DE OLIVEIRA,0.077961,0.023256,0.230902
1,1097837,0.000000,GABRIEL CIRAC MENDES DE SOUZA,0.200000,0.324128,0.137238
10,1682528,0.000000,THATYANA DE FARIA PIOLA SERAPHIM,0.547619,0.043478,0.122567
22,2195502,0.191912,GUSTAVO DELLA COLLETTA,0.049451,0.028068,0.114266
20,2127183,0.155556,BRUNO TARDIOLE KUEHNE,0.072482,0.050089,0.107301
29,2642873,0.121795,KLEBER ROBERTO DA SILVA SANTOS,0.139683,0.027983,0.097229


In [ ]:
def criar_candidatos_reranking(
    ranking,
    professores_dados,
    relacao_professor_trabalho,
    trabalhos_por_id,
    relacao_professor_disciplina,
    disciplinas_por_id,
    n_trabalhos=3,
    n_disciplinas=3
):

    candidatos = []

    # Obter professores pelo siape

    professores_por_id = {
        str(professor["siape"]): professor
        for professor in professores_dados
    }

    for _, candidato in ranking.iterrows():

        professor_id = str(candidato["professor_id"])

        #  busca a informação do perfil do professor

        perfil = professores_por_id.get(professor_id)

        if perfil is None:
            continue

        texto_perfil = montar_texto_perfil_professor(perfil)

        # busca informação dos trabalhos

        trabalhos_do_professor = (
            relacao_professor_trabalho[
                relacao_professor_trabalho["professor_id"].astype(str)
                == professor_id
            ]
            .sort_values("score", ascending=False)
            .head(n_trabalhos)
        )

        textos_trabalhos = []
        trabalhos_reranking = []

        for _, relacao in trabalhos_do_professor.iterrows():

            trabalho_id = str(relacao["objeto_id"])

            trabalho = trabalhos_por_id.get(trabalho_id)

            if trabalho is None:
                continue

            texto_trabalho = trabalho.get("texto_embedding")

            if texto_trabalho:
                textos_trabalhos.append(texto_trabalho)

            trabalhos_reranking.append({
                "trabalho_id": trabalho_id,
                "titulo": trabalho.get("metadata", {}).get("titulo"),
                "texto": texto_trabalho,
                "score": relacao.get("score")
            })

        # busca informações das disciplinas
        disciplinas_do_professor = (
            relacao_professor_disciplina[
                relacao_professor_disciplina["professor_id"].astype(str)
                == professor_id
            ]
            .sort_values("score", ascending=False)
            .head(n_disciplinas)
        )

        textos_disciplinas = []
        disciplinas_reranking = []

        for _, relacao in disciplinas_do_professor.iterrows():

            disciplina_id = str(relacao["objeto_id"])

            disciplina = disciplinas_por_id.get(disciplina_id)

            if disciplina is None:
                continue

            texto_disciplina = disciplina.get("texto_embedding")

            if texto_disciplina:
                textos_disciplinas.append(texto_disciplina)

            disciplinas_reranking.append({
                "disciplina_id": disciplina_id,
                "nome": disciplina.get("metadata", {}).get("nome"),
                "texto": texto_disciplina,
                "score": relacao.get("score")
            })

        texto_reranking = texto_perfil

        if textos_trabalhos:
            texto_reranking += (
                "\n\nTrabalhos relevantes:\n"
                + "\n\n".join(textos_trabalhos)
            )

        if textos_disciplinas:
            texto_reranking += (
                "\n\nDisciplinas relevantes:\n"
                + "\n\n".join(textos_disciplinas)
            )


        candidatos.append({
            "professor_id": professor_id,
            "professor_nome": candidato["professor_nome"],

            # Scores
            "work_score": candidato["work_score"],
            "professor_score": candidato["professor_score"],
            "disciplina_score": candidato["disciplina_score"],
            "score_final": candidato["score_final"],

            "texto_perfil": texto_perfil,
            "texto_reranking": texto_reranking,

            "trabalhos_relevantes": trabalhos_reranking,

            "disciplinas_relevantes": disciplinas_reranking
        })

    return candidatos

In [363]:
trabalhos_por_id = {
    str(trabalho["id"]): trabalho
    for trabalho in ics_perfis_busca
}

In [364]:
disciplina_por_id = {
    str(trabalho["id"]): trabalho
    for trabalho in disciplinas_perfil_busca
}

In [490]:
professores_para_reranking = criar_candidatos_reranking(candidatos_reranking, professores_dados,resultado_professor_trabalho_unico,trabalhos_por_id,resultado_professor_disciplina,disciplina_por_id)

In [491]:
professores_para_reranking

[{'professor_id': '1732597',
  'professor_nome': 'RODRIGO MAXIMIANO ANTUNES DE ALMEIDA',
  'work_score': 0.0,
  'professor_score': 0.4761905,
  'disciplina_score': 0.6170635,
  'score_final': 0.28035715,
  'texto_perfil': 'Resumo: Possui graduação em Engenharia de Controle e Automação pela Universidade Federal de Itajubá (2007), mestrado em Engenharia Elétrica pela Universidade Federal de Itajubá (2009) e doutorado em Engenharia Elétrica pela Universidade Federal de Itajubá (2013). Atualmente é professor adjunto da Universidade Federal de Itajubá. Leciona nas áreas de eletrônica, interface e periféricos, sistemas embarcados e sistemas operacionais. Suas pesquisas incluem tópicos sobre desenvolvimento de hardware, RTOS, segurança e usabilidade de sistemas eletrônicos. Trabalha em projetos de sistemas críticos onde confiabilidade e estabilidade são requisitos, seja do ponto de vista do sistema ou da interação do usuário com o equipamento\nFormação Profissional Acadêmica: Engenheiro de Co

Reranking (com a mesma lógica da busca cada conteúdo e rankeado e depois ligado a seu professor correspondente)

In [384]:
reranker = FlagReranker('BAAI/bge-reranker-v2-m3', use_fp16=True, normalize=True)

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

In [493]:
def criar_candidatos_reranking_v2(
    ranking,
    professores_dados,
    relacao_professor_trabalho,
    trabalhos_por_id,
    relacao_professor_disciplina,
    disciplinas_por_id,
    n_trabalhos=3,
    n_disciplinas=3
):

    candidatos = []

    # -----------------------------------
    # 1. Indexar professores pelo SIAPE
    # -----------------------------------

    professores_por_id = {
        str(professor["siape"]): professor
        for professor in professores_dados
    }

    # -----------------------------------
    # 2. Percorrer professores do ranking
    # -----------------------------------

    for _, candidato in ranking.iterrows():

        professor_id = str(candidato["professor_id"])

        perfil = professores_por_id.get(professor_id)

        if perfil is None:
            continue

        # -----------------------------------
        # 3. Perfil do professor
        # -----------------------------------

        texto_perfil = montar_texto_perfil_professor(perfil)

        if texto_perfil:

            candidatos.append({
                "professor_id": professor_id,
                "tipo": "professor",
                "item_id": professor_id,

                "texto": texto_perfil,

                "score_retrieval": candidato["professor_score"]
            })

        # -----------------------------------
        # 4. Trabalhos do professor
        # -----------------------------------

        trabalhos_do_professor = (
            relacao_professor_trabalho[
                relacao_professor_trabalho["professor_id"].astype(str)
                == professor_id
            ]
            .sort_values("score", ascending=False)
            .head(n_trabalhos)
        )

        for _, relacao in trabalhos_do_professor.iterrows():

            trabalho_id = str(relacao["objeto_id"])

            trabalho = trabalhos_por_id.get(trabalho_id)

            if trabalho is None:
                continue

            texto_trabalho = trabalho.get("texto_embedding")

            if not texto_trabalho:
                continue

            candidatos.append({
                "professor_id": professor_id,
                "tipo": "trabalho",
                "item_id": trabalho_id,

                "texto": texto_trabalho,

                "titulo": trabalho.get(
                    "metadata", {}
                ).get("titulo"),

                "score_retrieval": relacao["score"]
            })

        # -----------------------------------
        # 5. Disciplinas do professor
        # -----------------------------------

        disciplinas_do_professor = (
            relacao_professor_disciplina[
                relacao_professor_disciplina["professor_id"].astype(str)
                == professor_id
            ]
            .sort_values("score", ascending=False)
            .head(n_disciplinas)
        )

        for _, relacao in disciplinas_do_professor.iterrows():

            disciplina_id = str(relacao["objeto_id"])

            disciplina = disciplinas_por_id.get(
                disciplina_id
            )

            if disciplina is None:
                continue

            texto_disciplina = disciplina.get(
                "texto_embedding"
            )

            if not texto_disciplina:
                continue

            candidatos.append({
                "professor_id": professor_id,
                "tipo": "disciplina",
                "item_id": disciplina_id,

                "texto": texto_disciplina,

                "nome": disciplina.get(
                    "metadata", {}
                ).get("nome"),

                "score_retrieval": relacao["score"]
            })

    return candidatos

In [497]:
conteudo_reranking = criar_candidatos_reranking_v2(candidatos_reranking, professores_dados,resultado_professor_trabalho_unico,trabalhos_por_id,resultado_professor_disciplina,disciplina_por_id)

In [498]:
conteudo_reranking

[{'professor_id': '1732597',
  'tipo': 'professor',
  'item_id': '1732597',
  'texto': 'Resumo: Possui graduação em Engenharia de Controle e Automação pela Universidade Federal de Itajubá (2007), mestrado em Engenharia Elétrica pela Universidade Federal de Itajubá (2009) e doutorado em Engenharia Elétrica pela Universidade Federal de Itajubá (2013). Atualmente é professor adjunto da Universidade Federal de Itajubá. Leciona nas áreas de eletrônica, interface e periféricos, sistemas embarcados e sistemas operacionais. Suas pesquisas incluem tópicos sobre desenvolvimento de hardware, RTOS, segurança e usabilidade de sistemas eletrônicos. Trabalha em projetos de sistemas críticos onde confiabilidade e estabilidade são requisitos, seja do ponto de vista do sistema ou da interação do usuário com o equipamento\nFormação Profissional Acadêmica: Engenheiro de Controle e Automação, Mestrado e Doutorado em Engenharia Eletrica\nÁreas de Interesse: Sistemas embarcados, eletrônica, ensino de engenha

In [500]:
pares = [
    [aluno_texto, candidato["texto"]]
    for candidato in conteudo_reranking
]

In [501]:
scores = reranker.compute_score(
    pares,
    normalize=True
)

In [502]:
for candidato, score in zip(conteudo_reranking, scores):

    candidato["rerank_score"] = float(score)

In [504]:
from collections import defaultdict

def agrupar_por_professor(candidatos):
    professores = defaultdict(lambda: {
        "professor_id": None,
        "professor": None,
        "trabalhos_relevantes": [],
        "disciplinas_relevantes": []
    })

    for candidato in candidatos:
        professor_id = candidato["professor_id"]

        professores[professor_id]["professor_id"] = professor_id

        if candidato["tipo"] == "professor":
            professores[professor_id]["professor"] = candidato

        elif candidato["tipo"] == "trabalho":
            professores[professor_id]["trabalhos_relevantes"].append(candidato)

        elif candidato["tipo"] == "disciplina":
            professores[professor_id]["disciplinas_relevantes"].append(candidato)

    return dict(professores)

In [510]:
professores_ranked_agrupados = agrupar_por_professor(conteudo_reranking)

In [524]:
def calcular_score_final_professor(dados):
    componentes = []

    if dados["professor_rerank_score"] is not None:
        componentes.append(
            (0.20, dados["professor_rerank_score"])
        )

    if dados["work_rerank_score"] is not None:
        componentes.append(
            (0.50, dados["work_rerank_score"])
        )

    if dados["disciplina_rerank_score"] is not None:
        componentes.append(
            (0.30, dados["disciplina_rerank_score"])
        )

    if not componentes:
        return 0.0

    soma_pesos = sum(
        peso for peso, score in componentes
    )

    soma_scores = sum(
        peso * score
        for peso, score in componentes
    )

    return soma_scores / soma_pesos

In [525]:
def calcular_score_ponderado(itens, n=3):
    if not itens:
        return None

    itens_ordenados = sorted(
        itens,
        key=lambda x: x["rerank_score"],
        reverse=True
    )[:n]

    pesos = list(range(len(itens_ordenados), 0, -1))

    numerador = sum(
        item["rerank_score"] * peso
        for item, peso in zip(itens_ordenados, pesos)
    )

    denominador = sum(pesos)

    return numerador / denominador

In [526]:
def calcular_scores_professor(professores):
    for professor_id, dados in professores.items():

        # Score do perfil do professor
        if dados["professor"] is not None:
            dados["professor_rerank_score"] = (
                dados["professor"]["rerank_score"]
            )
        else:
            dados["professor_rerank_score"] = None

        # Score dos trabalhos
        dados["work_rerank_score"] = calcular_score_ponderado(
            dados["trabalhos_relevantes"],
            n=3
        )

        # Score das disciplinas
        dados["disciplina_rerank_score"] = calcular_score_ponderado(
            dados["disciplinas_relevantes"],
            n=3
        )

    return professores

In [536]:
professores_ranked_agrupados

{'1732597': {'professor_id': '1732597',
  'professor': {'professor_id': '1732597',
   'tipo': 'professor',
   'item_id': '1732597',
   'texto': 'Resumo: Possui graduação em Engenharia de Controle e Automação pela Universidade Federal de Itajubá (2007), mestrado em Engenharia Elétrica pela Universidade Federal de Itajubá (2009) e doutorado em Engenharia Elétrica pela Universidade Federal de Itajubá (2013). Atualmente é professor adjunto da Universidade Federal de Itajubá. Leciona nas áreas de eletrônica, interface e periféricos, sistemas embarcados e sistemas operacionais. Suas pesquisas incluem tópicos sobre desenvolvimento de hardware, RTOS, segurança e usabilidade de sistemas eletrônicos. Trabalha em projetos de sistemas críticos onde confiabilidade e estabilidade são requisitos, seja do ponto de vista do sistema ou da interação do usuário com o equipamento\nFormação Profissional Acadêmica: Engenheiro de Controle e Automação, Mestrado e Doutorado em Engenharia Eletrica\nÁreas de Inte

In [527]:
professores_scores = calcular_scores_professor(
    professores_ranked_agrupados
)

In [528]:
for professor_id, dados in professores_scores.items():

    dados["rerank_score"] = (
        calcular_score_final_professor(dados)
    )

In [529]:
ranking_final = sorted(
    professores_scores.values(),
    key=lambda x: x["rerank_score"],
    reverse=True
)

In [530]:
ranking_final

[{'professor_id': '1967206',
  'professor': {'professor_id': '1967206',
   'tipo': 'professor',
   'item_id': '1967206',
   'texto': 'Resumo: Graduou-se em Engenharia Elétrica (1995) pela Universidade Federal de Itajubá (UNIFEI). Concluiu mestrado em Engenharia Elétrica na área de Controle, Automação e Microinformática (1998), pela Universidade Federal de Santa Catarina (UFSC) e doutorado em Computação (2009) pela Universidade Federal Fluminense (UFF), na área de Sistemas Paralelos e Distribuídos. Realizou um ano de pesquisas, em doutorado sanduíche, na Universidade de Pittsburgh, EUA, no laboratório Power-Aware Real-Time Systems. Atuou como engenheiro eletrônico de sistemas embarcados para a empresa Di-elétrons Ltda em Belo Horizonte - MG, de 1999 a 2003. Foi professor de Introdução à Computação e Circuitos Eletro-Eletrônicos na UFRJ, campus de Macaé (temporário 20h nível Adjunto). Foi também engenheiro eletrônico de controle e automação da PETROBRAS em Macaé/RJ entre 2008 e 2014. Tem

In [531]:
def adicionar_nomes_professores(ranking_final, ranking):
    nomes = (
        ranking[
            ["professor_id", "professor_nome"]
        ]
        .drop_duplicates("professor_id")
        .set_index("professor_id")["professor_nome"]
        .to_dict()
    )

    for professor in ranking_final:
        professor["professor_nome"] = nomes.get(
            professor["professor_id"],
            professor["professor_id"]
        )

    return ranking_final

In [532]:
ranking_final = adicionar_nomes_professores(ranking_final,score_final_professores)

In [534]:
def imprimir_ranking(ranking_final, aluno_texto):

    print("CONSULTA DO ALUNO")
    print(aluno_texto)

    print("\n" + "=" * 100)
    print("RANKING COMPLETO")
    print("=" * 100)

    for i, professor in enumerate(ranking_final, start=1):

        print("=" * 100)

        print(
            f"{i}º - {professor['professor_nome']}"
        )

        print(
            f"Rerank score: "
            f"{professor['rerank_score']:.6f}"
        )

        print(
            f"Score inicial: "
            f"{professor['professor']['score_retrieval']:.6f}"
            if professor["professor"] is not None
            else "Score inicial: N/A"
        )

        # ==================================================
        # TRABALHOS
        # ==================================================

        print("\nTrabalhos relevantes:")

        trabalhos = professor["trabalhos_relevantes"]

        if not trabalhos:
            print("Nenhum trabalho relevante encontrado.")
        else:
            trabalhos = sorted(
                trabalhos,
                key=lambda x: x["rerank_score"],
                reverse=True
            )

            for j, trabalho in enumerate(trabalhos, start=1):

                titulo = trabalho.get("titulo")

                if not titulo:
                    titulo = trabalho["texto"].split("\n")[0]

                    if titulo.startswith("Título:"):
                        titulo = titulo.replace(
                            "Título:", "", 1
                        ).strip()

                print(f"\n  {j}. {titulo}")

                print(
                    f"     Score do trabalho: "
                    f"{trabalho['rerank_score']:.6f}"
                )

        # ==================================================
        # DISCIPLINAS
        # ==================================================

        print("\nDisciplinas relevantes:")

        disciplinas = professor["disciplinas_relevantes"]

        if not disciplinas:
            print("Nenhuma disciplina relevante encontrada.")
        else:
            disciplinas = sorted(
                disciplinas,
                key=lambda x: x["rerank_score"],
                reverse=True
            )

            for j, disciplina in enumerate(disciplinas, start=1):

                nome = disciplina.get("nome")

                if not nome:
                    nome = disciplina["texto"].split("\n")[0]

                    if nome.startswith("Nome:"):
                        nome = nome.replace(
                            "Nome:", "", 1
                        ).strip()

                print(f"\n  {j}. {nome}")

                print(
                    f"     Score da disciplina: "
                    f"{disciplina['rerank_score']:.6f}"
                )

In [535]:
imprimir_ranking(ranking_final[:10],aluno_texto)

CONSULTA DO ALUNO
Gostaria de desenvolver um projeto com inteligência artificial e programação embarcada

RANKING COMPLETO
1º - LUCIANO BERTINI
Rerank score: 0.056751
Score inicial: 0.500000

Trabalhos relevantes:
Nenhum trabalho relevante encontrado.

Disciplinas relevantes:

  1. PROGRAMAÇÃO EMBARCADA
     Score da disciplina: 0.107031

  2. LABORATÓRIO DE PROGRAMAÇÃO EMBARCADA
     Score da disciplina: 0.054940
2º - DECIO RENNO DE MENDONCA FARIA
Rerank score: 0.054642
Score inicial: 0.350000

Trabalhos relevantes:
Nenhum trabalho relevante encontrado.

Disciplinas relevantes:

  1. PROGRAMAÇÃO EMBARCADA
     Score da disciplina: 0.107031

  2. LABORATÓRIO DE PROGRAMAÇÃO EMBARCADA
     Score da disciplina: 0.054940
3º - RODRIGO MAXIMIANO ANTUNES DE ALMEIDA
Rerank score: 0.051538
Score inicial: 0.476191

Trabalhos relevantes:
Nenhum trabalho relevante encontrado.

Disciplinas relevantes:

  1. PROGRAMAÇÃO EMBARCADA
     Score da disciplina: 0.107031

  2. PROGRAMAÇÃO PARA SISTEMAS EMB